# B1 — open-vocabulary frontend swap (Colab)

Swaps the YOLOv8n-COCO detector for **YOLO-World**, prompted with Replica's
own class list, and re-scores under ConceptGraphs' protocol. Everything
downstream (CLIP relabelling, the 32 KB trace, the scorer) is untouched —
this changes **one stage**.

The script (`collab_tasks/scripts/embed_crops_openvocab.py`) is written and
CPU-verified against real room0 data: config cloning, vocab reading and
`load_sequence` all pass. **The GPU half — actually running YOLO-World +
CLIP over ~2000 frames — has never been executed anywhere.** This notebook
is that first real run, so treat every stage as unproven until it prints.

**Runtime:** any GPU. Unlike the ConceptGraphs notebook this needs no SAM
ViT-H, so **T4 is sufficient** — YOLO-World-s and CLIP ViT-B/32 are small.
L4 is faster but not required. CPU will work and be painfully slow.

**Safety:** writes only to `replica_<scene>_openvocab` config and output
paths. The 0.091-mAcc YOLO-COCO baseline cited in the paper draft is never
read for writing, never overwritten.

**Pre-registered expectation (from the brief — state it before running,
do not retrofit after):** mAcc under `n_exclude 6` should rise materially
above **0.091**, plausibly into **0.2–0.4** on room0. If it does *not*
rise, that is equally reportable: it would mean the memory is a second,
independent bottleneck. ConceptGraphs' published number is **0.406**.

## 0 · Runtime check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout
      or 'nvidia-smi: no GPU')
if not torch.cuda.is_available():
    print('NO GPU — Runtime > Change runtime type > T4 (or L4). '
          'The run works on CPU but takes many hours.')
else:
    print(f'{torch.cuda.get_device_name(0)} | torch {torch.__version__}')

## 1 · Drive workspace

Outputs are small and valuable (a CSV, an embedding tensor, a scores.json
per scene); the raw scene frames are large and re-fetchable. Only the
former go to Drive.

In [ ]:
from google.colab import drive; from pathlib import Path; import os
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/ssnslam_colab')
WORKSPACE = DRIVE / 'workspace'; HANDOFF = DRIVE / 'handoff'
for p in (WORKSPACE, HANDOFF): p.mkdir(parents=True, exist_ok=True)
print(WORKSPACE)

## 2 · Repo + deps

In [ ]:
import subprocess, os, shutil
from pathlib import Path
REPO_URL = 'https://github.com/SynapticScotsman/Semantic-Spiking-Neural-SLAM-2023.git'
BRANCH = 'results-sites'
# repo on the LOCAL VM disk: a git tree on Drive-FUSE goes stale (fetch ok,
# 'reset --hard origin/BRANCH' exit 128) and is slow. ~50 MB — a fresh
# shallow clone costs seconds and guarantees the exact branch tip.
REPO_DIR = Path('/content/Semantic-Spiking-Neural-SLAM-2023')
if (REPO_DIR / '.git').exists():
    r = subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    if r.returncode == 0:
        r = subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','FETCH_HEAD'])
    if r.returncode != 0:
        print('existing clone unusable — re-cloning'); shutil.rmtree(REPO_DIR)
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                    REPO_URL,str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# outputs -> Drive. NOTE: the repo TRACKS outputs/ (committed html/json),
# so a fresh clone always has a real directory here — an exists() check
# would silently skip the link and every result would die with the VM.
# Merge the tracked files into Drive (-n never clobbers a newer copy),
# then replace the directory with the link.
tgt = WORKSPACE / 'outputs'; tgt.mkdir(parents=True, exist_ok=True)
if not os.path.islink('outputs'):
    if os.path.isdir('outputs'):
        subprocess.run(['cp','-rn','outputs/.',str(tgt)], check=True)
        shutil.rmtree('outputs')
    os.symlink(tgt, 'outputs')
# raw frames stay on local VM disk: Drive-FUSE makes the 5 GB fetch and the
# per-frame read loop several times slower, and the fetch is resumable.
local = Path('/content/replica_data'); local.mkdir(parents=True, exist_ok=True)
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/replica'): os.symlink(local, 'data/replica')

!pip -q install "ultralytics>=8.1" transformers scipy 2>&1 | tail -2
!apt-get -qq install -y aria2 > /dev/null 2>&1 || true
print('repo ready at', os.getcwd())
print('outputs ->', os.path.realpath('outputs'), '(symlink:', os.path.islink('outputs'), ')')
print('data/replica ->', os.path.realpath('data/replica'), '(local, ephemeral)')

## 3 · Scene data + ground truth

B1 needs two things the swap itself does not produce:

- the raw frames/poses (`prepare_replica.py`), and
- `outputs/replica_<scene>/gt_instances.json` — the script reads the **scored
  class list straight out of the GT**, so the detector is prompted with
  exactly what will be judged rather than a hand-written list.

Both are skip-if-present. This cell also defines the streaming `sh()` helper
used by every later stage: it mirrors output to a log on Drive, so progress
is readable from drive.google.com without touching this page.

In [ ]:
SCENE = 'room0'      # ConceptGraphs' own demo scene — start here
CONF  = 0.15         # open-vocab detectors are noisier than COCO YOLO

import os, subprocess, time, datetime, threading
from pathlib import Path
assert os.path.exists('vsa_cognitive_mapping'), (
    'Kernel is not inside the repo (runtime restarted?). Re-run cells 1-2.')
assert 'WORKSPACE' in globals(), 'run the Drive cell first'

LOG = f'{WORKSPACE}/b1_log_{SCENE}.txt'
def _log(line):
    with open(LOG, 'a') as f: f.write(line + chr(10))
def sh(c, cwd=None):
    cmd = ' '.join(map(str, c))
    stamp = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{stamp}] $ {cmd}', flush=True); _log(f'[{stamp}] $ {cmd}')
    t0 = time.time(); last = [t0]
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    p = subprocess.Popen(c, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors='replace', env=env, cwd=cwd)
    def pulse():
        while p.poll() is None:
            time.sleep(30)
            if p.poll() is None and time.time() - last[0] > 300:
                m = int((time.time() - t0) / 60)
                msg = f'  … still running ({m} min, no new output — normal for long model passes)'
                print(msg, flush=True); _log(msg); last[0] = time.time()
    threading.Thread(target=pulse, daemon=True).start()
    with open(LOG, 'a') as f:
        for ln in p.stdout:
            ln = ln.rstrip()
            print(ln, flush=True); f.write(ln + chr(10)); f.flush(); last[0] = time.time()
    rc = p.wait()
    done = f'  exit {rc} in {time.time()-t0:.0f}s'
    print(done, flush=True); _log(done)
    if rc != 0:
        raise RuntimeError(f'stage failed: {cmd} — full output in {LOG}')
    return True

if not Path(f'data/replica/{SCENE}/poses.csv').exists():
    sh(['python','tools/prepare_replica.py','--scene',SCENE])

# The GT extractor writes TWO things in different lifetimes:
#   outputs/replica_<scene>/gt_instances.json   -> Drive, survives the VM
#   data/replica/<scene>_vmap/*                 -> local disk, dies with it
# Guarding on the Drive half alone silently skips the extractor on a fresh
# VM whose Drive already has the json, and stage 4 then dies on a missing
# info_semantic.json. Require BOTH.
import re, hashlib, shutil
vscene = re.sub(r'(\D)(\d+)$', r'\1_\2', SCENE)
GT   = f'outputs/replica_{SCENE}/gt_instances.json'
INFO = f'data/replica/{vscene}_vmap/info_semantic.json'
if not (Path(GT).exists() and Path(INFO).exists()):
    # The extractor REWRITES gt_instances.json — the cited baseline GT.
    # Hash it, run, verify byte-identical, restore if not.
    before = (hashlib.sha256(open(GT,'rb').read()).hexdigest()
              if os.path.exists(GT) else None)
    if before:
        shutil.copy(GT, GT + '.baseline_backup')
        print(f'baseline GT sha256 {before[:16]}… backed up')
    sh(['python','tools/replica_gt_from_renders.py','--scene',SCENE])
    if before:
        after = hashlib.sha256(open(GT,'rb').read()).hexdigest()
        if after == before:
            print(f'baseline GT unchanged ({after[:16]}…) — deterministic')
            os.remove(GT + '.baseline_backup')
        else:
            shutil.move(GT + '.baseline_backup', GT)
            print(f'WARNING baseline GT CHANGED ({before[:16]}… -> {after[:16]}…)'
                  ' — restored from backup. Investigate before scoring.')
print(f'prerequisites present for {SCENE}: GT={Path(GT).exists()} '
      f'vmap_info={Path(INFO).exists()}')

## 4 · The swap itself

`embed_crops_openvocab.py` clones the scene config to
`replica_<scene>_openvocab.json`, prompts YOLO-World with the GT class list,
and writes `detections_crops.csv` + `crop_embeddings_openvocab.pt` into a
brand-new output directory.

**This is the never-before-executed part.** Expect the first failure here if
there is one — most likely an `ultralytics` API drift around
`model.set_classes()`, or the `yolov8s-worldv2.pt` download. The script
raises loudly rather than continuing on empty detections.

In [ ]:
t0 = time.time()
sh(['python','collab_tasks/scripts/embed_crops_openvocab.py',
    '--scene',SCENE,'--conf',str(CONF)])
print(f'B1 frontend wall-clock: {(time.time()-t0)/60:.1f} min')

## 5 · Score it — same scorer, same GT, only the frontend changed

`--gt-scene {SCENE}` points stage 4 at the baseline's existing ground truth
(same physical scene) while keeping every output under the `_openvocab`
paths. This is what makes the comparison apples-to-apples.

Two messages here are **expected, not failures**:

- `conceptgraphs: cg_labels.npz missing — skipped` — this run scores our
  memory only; ConceptGraphs' own labels come from the separate
  ConceptGraphs notebook. Only the `vsa` row is meaningful here.
- `reusing .../{SCENE}/eval_points.npz` if the baseline was run on this VM.
  If it wasn't, stage 4 rebuilds the eval points from the vMAP renders
  fetched in cell 3 — same GT either way, just slower the first time.

In [ ]:
sh(['python','-m','vsa_cognitive_mapping.object_grounding',
    '--dataset',f'vsa_cognitive_mapping/configs/replica_{SCENE}_openvocab.json',
    '--gt-json',f'outputs/replica_{SCENE}/gt_instances.json','--relocalize'])
sh(['python','student_gpu_package/04_vsa_labels.py',
    '--scene',f'{SCENE}_openvocab','--gt-scene',SCENE])

# Safety net for older checkouts: before the 04-copies-eval-points fix,
# stage 4 reused handoff/<gt_scene>/eval_points.npz in place while stage 5
# looked in handoff/<scene>/ — every alternate-frontend run died here.
# No-op once the clone carries the fix.
_src = f'student_gpu_package/handoff/{SCENE}/eval_points.npz'
_dst = f'student_gpu_package/handoff/{SCENE}_openvocab/eval_points.npz'
if not os.path.exists(_dst) and os.path.exists(_src):
    os.makedirs(os.path.dirname(_dst), exist_ok=True)
    shutil.copy(_src, _dst)
    print(f'(patched older checkout: copied eval_points.npz -> {_dst})')

sh(['python','student_gpu_package/05_score.py','--scene',f'{SCENE}_openvocab'])

import json
scores = json.load(open(f'student_gpu_package/handoff/{SCENE}_openvocab/scores.json'))
print(json.dumps(scores, indent=2))

## 6 · Self-check before believing the number

The brief says to check the per-class breakdown for degeneracy — but
`05_score.py` writes **aggregates only** (`their_protocol`, `all_classes`,
`unmatched_frac`); there is no per-class field in `scores.json`. So this
cell checks degeneracy at the two places the evidence actually exists:

1. the **detector's** class distribution (`detections_crops.csv`), and
2. the **predicted labels** actually scored (`vsa_labels.npz`).

Every prediction collapsing onto one catch-all class is the silent-failure
mode the brief warns about — it can produce a plausible-looking mAcc.

In [ ]:
import csv, collections, numpy as np, json, os

OUT = f'outputs/replica_{SCENE}_openvocab'
rows = list(csv.DictReader(open(f'{OUT}/detections_crops.csv')))
det = collections.Counter(r['class_name'] for r in rows)
print(f'detections: {len(rows)} crops, {len(det)} distinct classes')
for name, n in det.most_common(12):
    print(f'   {n:6d}  {n/len(rows):5.1%}  {name}')
top_share = det.most_common(1)[0][1] / len(rows) if rows else 1.0

d = f'student_gpu_package/handoff/{SCENE}_openvocab'
lab_path = os.path.join(d, 'vsa_labels.npz')
if os.path.exists(lab_path):
    Z = np.load(lab_path, allow_pickle=True)
    pred = collections.Counter(Z['pred_class'].astype(str))
    print(f'\nscored labels: {len(Z["pred_class"])} points, {len(pred)} distinct')
    for name, n in pred.most_common(10):
        print(f'   {n:8d}  {n/len(Z["pred_class"]):5.1%}  {name}')

s = json.load(open(f'{d}/scores.json'))
v = s['vsa']['their_protocol']
print(f"\nB1 (open-vocab): mAcc {v['mAcc']:.3f}  F-mIoU {v['fmiou']:.3f}"
      f"  | unmatched {s['vsa']['unmatched_frac']:.1%}")
print(f"baseline (YOLO-COCO, measured): mAcc 0.091")
print(f"ConceptGraphs (published):      mAcc 0.406")

# The pre-registration is a BAND (0.2-0.4), not "anything above baseline".
# Scoring it as ">0.091 == success" would let a modest rise be written up as
# a confirmed hypothesis; state which of the three outcomes actually happened.
BASELINE, BAND_LO, BAND_HI, CG_PUB = 0.091, 0.20, 0.40, 0.406
m = v['mAcc']
print('\nVERDICT (against the pre-registered band 0.20-0.40, not just the baseline)')
if len(det) < 3 or top_share > 0.9:
    print(f'  DEGENERATE frontend — {top_share:.0%} of detections are one class. '
          'The number above is not trustworthy; try a lower --conf and check '
          'the printed vocab was not empty/garbled.')
elif m >= BAND_LO:
    print(f'  CONFIRMED: {m:.3f} is inside the pre-registered band '
          f'({BAND_LO}-{BAND_HI}), +{m-BASELINE:.3f} over baseline. '
          'The frontend was the bottleneck, as predicted.')
elif m > BASELINE:
    print(f'  PARTIAL: {m:.3f} rose over the {BASELINE} baseline '
          f'(+{m-BASELINE:.3f}, {(m/BASELINE-1)*100:.0f}% relative) but landed '
          f'BELOW the pre-registered {BAND_LO}-{BAND_HI}. The frontend is A '
          'bottleneck, not the only one — report the miss, do not round the '
          'claim up to "confirmed".')
else:
    print(f'  NOT SUPPORTED: {m:.3f} did not rise above {BASELINE}. Per the '
          'brief this is equally reportable — it points at the memory as a '
          'second, independent bottleneck. Report as measured; do not tune '
          'toward the number.')
print(f'  gap to ConceptGraphs ({CG_PUB}): {CG_PUB-m:.3f}')

## 7 · Qualitative artifact

A number ships with something inspectable, always. This is the same
exporter the local pipeline uses, pointed at the openvocab config and the
untouched baseline GT: a browsable page of placed detections with photos,
per-class consistency and distance-to-truth.

In [ ]:
INSPECTOR = f'outputs/replica_{SCENE}_openvocab/grounding_inspector.html'
sh(['python','tools/export_grounding_inspector.py',
    '--dataset',f'vsa_cognitive_mapping/configs/replica_{SCENE}_openvocab.json',
    '--gt-json',f'outputs/replica_{SCENE}/gt_instances.json',
    '--out',INSPECTOR])
print(f'\n{INSPECTOR}  ({os.path.getsize(INSPECTOR)/1e6:.1f} MB)')

## 8 · Save the handoff to Drive

In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M')
dst = HANDOFF / f'{SCENE}_openvocab_{stamp}'
shutil.copytree(f'student_gpu_package/handoff/{SCENE}_openvocab', dst, dirs_exist_ok=True)
for f in ('detections_crops.csv','crop_embeddings_openvocab.pt',
          'grounding_inspector.html'):
    p = f'outputs/replica_{SCENE}_openvocab/{f}'
    if os.path.exists(p): shutil.copy(p, dst)
with open(dst / 'run_notes.txt','w') as f:
    f.write(f'detector: yolov8s-worldv2 (open-vocab)\nconf: {CONF}\n'
            f'scene: {SCENE}\ngt-scene: {SCENE} (baseline GT, untouched)\n')
!pip list 2>/dev/null | grep -Ei 'ultralytics|torch|transformers|clip' >> {dst}/run_notes.txt
print('saved to', dst)
!du -sh {dst}

## 9 · Other scenes

Change `SCENE` in cell 3 and re-run cells 3–8. The frontend is the only
GPU-heavy stage, and it is far cheaper than the ConceptGraphs pipeline —
no SAM, no pytorch3d, no 3D fusion.

Scenes: `room0 room1 room2 office0 office1 office2 office3 office4`.

**What to send back per scene:** `detections_crops.csv`,
`crop_embeddings_openvocab.pt`, `scores.json`, plus the detector
checkpoint, the `--conf` used, and wall-clock. If a scene fails, send its
log from `b1_log_<scene>.txt` on Drive rather than a guessed fix.